# Setup and Auth

Use this notebook to verify imports, configure the base API URL, and optionally test authenticated endpoints.

In [ ]:
import os
from pprint import pprint

import requests
from dotenv import load_dotenv

load_dotenv()

BASE_URL = os.getenv("NMRXIV_BASE_URL", "https://nmrxiv.org").rstrip("/")
API_BASE = f"{BASE_URL}/api"

session = requests.Session()
session.headers.update({
    "Accept": "application/json",
    "Content-Type": "application/json",
})

BASE_URL, API_BASE

In [ ]:
def api_request(method, path, **kwargs):
    url = f"{API_BASE}{path}"
    response = session.request(method, url, timeout=30, **kwargs)
    print(f"{response.request.method} {response.url} -> {response.status_code}")
    try:
        payload = response.json()
    except ValueError:
        payload = response.text
    if not response.ok:
        pprint(payload)
        response.raise_for_status()
    return payload


def extract_token(payload):
    if not isinstance(payload, dict):
        return None
    token = payload.get("access_token") or payload.get("token")
    if token:
        return token
    data = payload.get("data")
    if isinstance(data, dict):
        return data.get("access_token") or data.get("token")
    return None

## Optional credentials

Copy `.env.example` to `.env` and fill in your account values if you want to test login. Public notebooks do not require this.

In [ ]:
email = os.getenv("NMRXIV_EMAIL")
password = os.getenv("NMRXIV_PASSWORD")

if email and password:
    login_response = api_request("POST", "/auth/login", json={"email": email, "password": password})
    pprint(login_response)
    token = extract_token(login_response)
    if token:
        session.headers.update({"Authorization": f"Bearer {token}"})
        print("Bearer token configured for this notebook session.")
    else:
        print("Login succeeded, but no token key was recognized. Inspect login_response above.")
else:
    print("No credentials found. Skipping login.")

In [ ]:
# Run this only after successful login.
if "Authorization" in session.headers:
    user_info = api_request("GET", "/auth/user/info")
    pprint(user_info)
else:
    print("Login first to call /auth/user/info.")

## Auth endpoints in Swagger

- `POST /api/auth/login`
- `GET /api/auth/logout`
- `POST /api/auth/register`
- `GET /api/auth/user/info`
- `GET /api/auth/email/resend`